In [ ]:
import os
import json
import shutil
import torch
import torch.nn as nn

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH = "/content/drive/MyDrive/Underwater-Image-Data-set-main"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
config_path = os.path.join(
    BASE_PATH,
    "Day19_Configuration.json"
)

with open(config_path, "r") as f:
    day19_config = json.load(f)

print("========== SELECTED CONFIGURATION ==========")

for key, value in day19_config.items():
    print(f"{key}: {value}")

========== SELECTED CONFIGURATION ==========
model: U-Net
dataset: Dataset V1
loss_function: L1Loss
optimizer: Adam
epochs: 5
selected_learning_rate: 0.0001
selected_validation_loss: 0.1759122337984002


In [ ]:
split_path = os.path.join(
    BASE_PATH,
    "Dataset_V1_frozen_split.json"
)

with open(split_path, "r") as f:
    split_data = json.load(f)

print("========== FROZEN DATASET ==========")

print(
    "Training samples:",
    len(split_data["train"])
)

print(
    "Validation samples:",
    len(split_data["validation"])
)

print(
    "Test samples:",
    len(split_data["test"])
)

========== FROZEN DATASET ==========
Training samples: 863
Validation samples: 184
Test samples: 186


In [ ]:
class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1
            ),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.down1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.conv3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.conv2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.conv1 = DoubleConv(
            64,
            32
        )

        self.final = nn.Conv2d(
            32,
            3,
            1
        )

    def forward(self, x):

        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        m = self.middle(p3)

        u3 = self.up3(m)

        u3 = torch.cat(
            [u3, d3],
            dim=1
        )

        u3 = self.conv3(u3)

        u2 = self.up2(u3)

        u2 = torch.cat(
            [u2, d2],
            dim=1
        )

        u2 = self.conv2(u2)

        u1 = self.up1(u2)

        u1 = torch.cat(
            [u1, d1],
            dim=1
        )

        u1 = self.conv1(u1)

        return torch.sigmoid(
            self.final(u1)
        )

In [ ]:
model_path = os.path.join(
    BASE_PATH,
    "Day17_Results",
    "best_unet_model.pth"
)

final_model = UNet().to(device)

final_model.load_state_dict(
    torch.load(
        model_path,
        map_location=device
    )
)

final_model.eval()

print("Final candidate loaded successfully.")

Final candidate loaded successfully.


In [ ]:
final_folder = os.path.join(
    BASE_PATH,
    "Final_Frozen_Model"
)

os.makedirs(
    final_folder,
    exist_ok=True
)

print("Final folder:")
print(final_folder)

Final folder:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Final_Frozen_Model


In [ ]:
final_model_path = os.path.join(
    final_folder,
    "final_unet_model.pth"
)

torch.save(
    final_model.state_dict(),
    final_model_path
)

print("FINAL MODEL SAVED")
print(final_model_path)

FINAL MODEL SAVED
/content/drive/MyDrive/Underwater-Image-Data-set-main/Final_Frozen_Model/final_unet_model.pth


In [ ]:
final_config = {

    "project_stage":
        "FINAL FROZEN MODEL",

    "model":
        "U-Net",

    "dataset":
        "Dataset V1",

    "dataset_split":
        "Dataset_V1_frozen_split.json",

    "input_size":
        [256, 256],

    "input_channels":
        3,

    "output_channels":
        3,

    "loss_function":
        "L1Loss",

    "optimizer":
        "Adam",

    "selected_learning_rate":
        day19_config[
            "selected_learning_rate"
        ],

    "training_epochs":
        day19_config["epochs"],

    "selected_validation_loss":
        day19_config[
            "selected_validation_loss"
        ],

    "model_file":
        "final_unet_model.pth",

    "preprocessing":
        "RGB conversion, resize to 256x256, normalize to [0,1]",

    "status":
        "FROZEN"
}

final_config_path = os.path.join(
    final_folder,
    "final_config.json"
)

with open(
    final_config_path,
    "w"
) as f:

    json.dump(
        final_config,
        f,
        indent=4
    )

print("Final configuration saved.")

Final configuration saved.


In [ ]:
final_split_path = os.path.join(
    final_folder,
    "Dataset_V1_frozen_split.json"
)

shutil.copy2(
    split_path,
    final_split_path
)

print(
    "Frozen dataset split copied."
)

Frozen dataset split copied.


In [ ]:
total_parameters = sum(
    p.numel()
    for p in final_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in final_model.parameters()
    if p.requires_grad
)

model_size_mb = (
    os.path.getsize(
        final_model_path
    ) / (1024 ** 2)
)

model_info = {

    "architecture":
        "U-Net",

    "total_parameters":
        total_parameters,

    "trainable_parameters":
        trainable_parameters,

    "model_size_MB":
        model_size_mb,

    "input_resolution":
        "256x256",

    "input_channels":
        3,

    "output_channels":
        3
}

print(
    json.dumps(
        model_info,
        indent=4
    )
)

{
    "architecture": "U-Net",
    "total_parameters": 1925667,
    "trainable_parameters": 1925667,
    "model_size_MB": 7.358994483947754,
    "input_resolution": "256x256",
    "input_channels": 3,
    "output_channels": 3
}


In [ ]:
model_info_path = os.path.join(
    final_folder,
    "model_info.json"
)

with open(
    model_info_path,
    "w"
) as f:

    json.dump(
        model_info,
        f,
        indent=4
    )

print(
    "Model information saved."
)

Model information saved.


In [ ]:
verification_model = UNet().to(device)

verification_model.load_state_dict(
    torch.load(
        final_model_path,
        map_location=device
    )
)

verification_model.eval()

print("================================")
print("FINAL MODEL VERIFICATION")
print("================================")

print(
    "Model loaded successfully: YES"
)

print(
    "Model status: FROZEN"
)

print(
    "Learning rate:",
    final_config["selected_learning_rate"]
)

print(
    "Validation loss:",
    final_config["selected_validation_loss"]
)

print(
    "Parameters:",
    total_parameters
)

print(
    "Model size:",
    round(model_size_mb, 2),
    "MB"
)

FINAL MODEL VERIFICATION
Model loaded successfully: YES
Model status: FROZEN
Learning rate: 0.0001
Validation loss: 0.1759122337984002
Parameters: 1925667
Model size: 7.36 MB


In [ ]:
print("================================")
print("FINAL PACKAGE")
print("================================")

for filename in os.listdir(
    final_folder
):

    filepath = os.path.join(
        final_folder,
        filename
    )

    size = os.path.getsize(
        filepath
    ) / 1024

    print(
        f"{filename:<35} "
        f"{size:.2f} KB"
    )

FINAL PACKAGE
final_unet_model.pth                7535.61 KB
final_config.json                   0.55 KB
Dataset_V1_frozen_split.json        285.87 KB
model_info.json                     0.22 KB


In [ ]:
from datetime import datetime

freeze_record = {

    "freeze_status":
        "FROZEN",

    "freeze_date":
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "model":
        "U-Net",

    "learning_rate":
        final_config[
            "selected_learning_rate"
        ],

    "validation_loss":
        final_config[
            "selected_validation_loss"
        ],

    "dataset":
        "Dataset V1",

    "split":
        "Frozen",

    "preprocessing":
        "Frozen",

    "further_training":
        "Not performed after freeze",

    "test_evaluation":
        "Reserved for Day 23"
}

freeze_path = os.path.join(
    final_folder,
    "FREEZE_RECORD.json"
)

with open(
    freeze_path,
    "w"
) as f:

    json.dump(
        freeze_record,
        f,
        indent=4
    )

print("Freeze record created.")
print(freeze_path)

Freeze record created.
/content/drive/MyDrive/Underwater-Image-Data-set-main/Final_Frozen_Model/FREEZE_RECORD.json
